# FOG Task-Category × Interaction-Depth Group Figures

This notebook is self-contained. It embeds every aggregated value used by the two group figures:

- six panels: Overall, Localization, Semantic, Cross-modal, Temporal, and Instant;
- four fixed depth bins: step 0, step 1, step 2, and step 3+;
- Frozen Qwen and Qwen + FOG;
- Action Type Match and Exact Action Match;
- sample counts, episode counts, means, and 90% episode-clustered bootstrap confidence intervals.

The embedded values were recomputed from the same paired 508-step / 142-episode held-out predictions using 10,000 episode-clustered bootstrap resamples (seed 20260826), with the 5th and 95th percentiles forming the 90% intervals. No raw log is required to reproduce these plots.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CATEGORY_ORDER = (
    ("overall", "Overall"),
    ("localization", "Localization"),
    ("semantic_understanding", "Semantic"),
    ("cross_modal_discrimination", "Cross-modal"),
    ("temporal_reasoning", "Temporal"),
    ("instant_response", "Instant"),
)
DEPTH_ORDER = ("0", "1", "2", "3+")
METRIC_NAMES = {
    "type_match": "Action Type Match",
    "exact_match": "Exact Action Match",
}
METHOD_NAMES = {
    "raw": "Frozen Qwen",
    "fog": "Qwen + FOG",
}
COLORS = {
    "raw": "#6B7280",
    "fog": "#007C91",
}

CONFIDENCE_LEVEL = 0.90
BOOTSTRAP_ITERATIONS = 10_000
BOOTSTRAP_SEED = 20260826

# Change this directory when a different export location is desired.
OUTPUT_DIR = Path.cwd() / "fog_group_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11.5,
    "axes.labelsize": 10.5,
    "legend.fontsize": 9,
    "figure.dpi": 120,
})


In [ ]:
# Complete data used by both figures. Values are proportions in [0, 1].
# CIs use 10,000 episode-clustered bootstrap resamples; 90% = [q0.05, q0.95].
DATA = json.loads(r'''{"overall":[{"depth":"0","steps":142,"episodes":142,"metrics":{"type_match":{"raw":{"mean":0.647887323943662,"ci90":[0.5774647887323944,0.7112676056338029]},"fog":{"mean":0.8028169014084507,"ci90":[0.7464788732394366,0.8591549295774648]},"fog_minus_raw":{"mean":0.15492957746478872,"ci90":[0.08450704225352113,0.22535211267605634]}},"exact_match":{"raw":{"mean":0.1619718309859155,"ci90":[0.11267605633802817,0.2112676056338028]},"fog":{"mean":0.3873239436619718,"ci90":[0.323943661971831,0.45774647887323944]},"fog_minus_raw":{"mean":0.22535211267605634,"ci90":[0.16901408450704225,0.2887323943661972]}}}},{"depth":"1","steps":141,"episodes":141,"metrics":{"type_match":{"raw":{"mean":0.5319148936170213,"ci90":[0.46099290780141844,0.6028368794326241]},"fog":{"mean":0.6453900709219859,"ci90":[0.5815602836879432,0.7092198581560284]},"fog_minus_raw":{"mean":0.11347517730496454,"ci90":[0.02127659574468085,0.20567375886524822]}},"exact_match":{"raw":{"mean":0.10638297872340426,"ci90":[0.06382978723404255,0.14893617021276595]},"fog":{"mean":0.3404255319148936,"ci90":[0.2765957446808511,0.40425531914893614]},"fog_minus_raw":{"mean":0.23404255319148937,"ci90":[0.1702127659574468,0.3049645390070922]}}}},{"depth":"2","steps":103,"episodes":103,"metrics":{"type_match":{"raw":{"mean":0.4368932038834951,"ci90":[0.3592233009708738,0.5145631067961165]},"fog":{"mean":0.7184466019417476,"ci90":[0.6407766990291263,0.7864077669902912]},"fog_minus_raw":{"mean":0.2815533980582524,"ci90":[0.17475728155339806,0.3883495145631068]}},"exact_match":{"raw":{"mean":0.11650485436893204,"ci90":[0.06796116504854369,0.17475728155339806]},"fog":{"mean":0.42718446601941745,"ci90":[0.34951456310679613,0.5048543689320388]},"fog_minus_raw":{"mean":0.3106796116504854,"ci90":[0.22330097087378642,0.39805825242718446]}}}},{"depth":"3+","steps":122,"episodes":62,"metrics":{"type_match":{"raw":{"mean":0.32786885245901637,"ci90":[0.25196850393700787,0.3937007874015748]},"fog":{"mean":0.6967213114754098,"ci90":[0.625,0.775708991298743]},"fog_minus_raw":{"mean":0.36885245901639346,"ci90":[0.2740740740740741,0.47619047619047616]}},"exact_match":{"raw":{"mean":0.11475409836065574,"ci90":[0.06194234102727854,0.16912330316742075]},"fog":{"mean":0.4672131147540984,"ci90":[0.39705882352941174,0.5545454545454546]},"fog_minus_raw":{"mean":0.3524590163934426,"ci90":[0.2519055696918292,0.4672897196261682]}}}}],"localization":[{"depth":"0","steps":27,"episodes":27,"metrics":{"type_match":{"raw":{"mean":0.8518518518518519,"ci90":[0.7407407407407407,0.9629629629629629]},"fog":{"mean":0.9259259259259259,"ci90":[0.8518518518518519,1.0]},"fog_minus_raw":{"mean":0.07407407407407407,"ci90":[-0.07407407407407407,0.2222222222222222]}},"exact_match":{"raw":{"mean":0.14814814814814814,"ci90":[0.037037037037037035,0.25925925925925924]},"fog":{"mean":0.48148148148148145,"ci90":[0.3333333333333333,0.6296296296296297]},"fog_minus_raw":{"mean":0.3333333333333333,"ci90":[0.18518518518518517,0.48148148148148145]}}}},{"depth":"1","steps":27,"episodes":27,"metrics":{"type_match":{"raw":{"mean":0.4444444444444444,"ci90":[0.2962962962962963,0.5925925925925926]},"fog":{"mean":0.7407407407407407,"ci90":[0.5925925925925926,0.8888888888888888]},"fog_minus_raw":{"mean":0.2962962962962963,"ci90":[0.07407407407407407,0.5185185185185185]}},"exact_match":{"raw":{"mean":0.07407407407407407,"ci90":[0.0,0.14814814814814814]},"fog":{"mean":0.48148148148148145,"ci90":[0.3333333333333333,0.6296296296296297]},"fog_minus_raw":{"mean":0.4074074074074074,"ci90":[0.2222222222222222,0.5925925925925926]}}}},{"depth":"2","steps":14,"episodes":14,"metrics":{"type_match":{"raw":{"mean":0.35714285714285715,"ci90":[0.14285714285714285,0.5714285714285714]},"fog":{"mean":0.5714285714285714,"ci90":[0.35714285714285715,0.7857142857142857]},"fog_minus_raw":{"mean":0.21428571428571427,"ci90":[-0.07142857142857142,0.5]}},"exact_match":{"raw":{"mean":0.14285714285714285,"ci90":[0.0,0.2857142857142857]},"fog":{"mean":0.42857142857142855,"ci90":[0.21428571428571427,0.6428571428571429]},"fog_minus_raw":{"mean":0.2857142857142857,"ci90":[0.0,0.5714285714285714]}}}},{"depth":"3+","steps":14,"episodes":6,"metrics":{"type_match":{"raw":{"mean":0.5,"ci90":[0.25,0.6666666666666666]},"fog":{"mean":0.7857142857142857,"ci90":[0.6,0.9333333333333333]},"fog_minus_raw":{"mean":0.2857142857142857,"ci90":[0.14285714285714285,0.5]}},"exact_match":{"raw":{"mean":0.14285714285714285,"ci90":[0.0,0.25]},"fog":{"mean":0.35714285714285715,"ci90":[0.2222222222222222,0.6]},"fog_minus_raw":{"mean":0.21428571428571427,"ci90":[0.058823529411764705,0.5]}}}}],"semantic_understanding":[{"depth":"0","steps":29,"episodes":29,"metrics":{"type_match":{"raw":{"mean":0.5172413793103449,"ci90":[0.3793103448275862,0.6551724137931034]},"fog":{"mean":0.8275862068965517,"ci90":[0.7241379310344828,0.9310344827586207]},"fog_minus_raw":{"mean":0.3103448275862069,"ci90":[0.13793103448275862,0.4827586206896552]}},"exact_match":{"raw":{"mean":0.1724137931034483,"ci90":[0.06896551724137931,0.3103448275862069]},"fog":{"mean":0.41379310344827586,"ci90":[0.27586206896551724,0.5517241379310345]},"fog_minus_raw":{"mean":0.2413793103448276,"ci90":[0.10344827586206896,0.3793103448275862]}}}},{"depth":"1","steps":28,"episodes":28,"metrics":{"type_match":{"raw":{"mean":0.5,"ci90":[0.35714285714285715,0.6428571428571429]},"fog":{"mean":0.6785714285714286,"ci90":[0.5357142857142857,0.8214285714285714]},"fog_minus_raw":{"mean":0.17857142857142858,"ci90":[0.0,0.35714285714285715]}},"exact_match":{"raw":{"mean":0.14285714285714285,"ci90":[0.03571428571428571,0.25]},"fog":{"mean":0.35714285714285715,"ci90":[0.21428571428571427,0.5]},"fog_minus_raw":{"mean":0.21428571428571427,"ci90":[0.07142857142857142,0.35714285714285715]}}}},{"depth":"2","steps":20,"episodes":20,"metrics":{"type_match":{"raw":{"mean":0.65,"ci90":[0.45,0.8]},"fog":{"mean":0.7,"ci90":[0.55,0.85]},"fog_minus_raw":{"mean":0.05,"ci90":[-0.2,0.3]}},"exact_match":{"raw":{"mean":0.15,"ci90":[0.05,0.3]},"fog":{"mean":0.35,"ci90":[0.2,0.5]},"fog_minus_raw":{"mean":0.2,"ci90":[0.0,0.4]}}}},{"depth":"3+","steps":34,"episodes":15,"metrics":{"type_match":{"raw":{"mean":0.35294117647058826,"ci90":[0.21739130434782608,0.4339622641509434]},"fog":{"mean":0.5588235294117647,"ci90":[0.43902439024390244,0.7391304347826086]},"fog_minus_raw":{"mean":0.20588235294117646,"ci90":[0.07142857142857142,0.47619047619047616]}},"exact_match":{"raw":{"mean":0.20588235294117646,"ci90":[0.058739495798319406,0.30303030303030304]},"fog":{"mean":0.4411764705882353,"ci90":[0.32075471698113206,0.6666666666666666]},"fog_minus_raw":{"mean":0.23529411764705882,"ci90":[0.04993902439024396,0.5652173913043478]}}}}],"cross_modal_discrimination":[{"depth":"0","steps":31,"episodes":31,"metrics":{"type_match":{"raw":{"mean":0.4838709677419355,"ci90":[0.3225806451612903,0.6451612903225806]},"fog":{"mean":0.6451612903225806,"ci90":[0.5161290322580645,0.7741935483870968]},"fog_minus_raw":{"mean":0.16129032258064516,"ci90":[0.0,0.3225806451612903]}},"exact_match":{"raw":{"mean":0.12903225806451613,"ci90":[0.03225806451612903,0.22580645161290322]},"fog":{"mean":0.2903225806451613,"ci90":[0.16129032258064516,0.41935483870967744]},"fog_minus_raw":{"mean":0.16129032258064516,"ci90":[0.03225806451612903,0.2903225806451613]}}}},{"depth":"1","steps":31,"episodes":31,"metrics":{"type_match":{"raw":{"mean":0.41935483870967744,"ci90":[0.2903225806451613,0.5806451612903226]},"fog":{"mean":0.5483870967741935,"ci90":[0.3870967741935484,0.6774193548387096]},"fog_minus_raw":{"mean":0.12903225806451613,"ci90":[-0.0967741935483871,0.3548387096774194]}},"exact_match":{"raw":{"mean":0.03225806451612903,"ci90":[0.0,0.0967741935483871]},"fog":{"mean":0.1935483870967742,"ci90":[0.0967741935483871,0.3225806451612903]},"fog_minus_raw":{"mean":0.16129032258064516,"ci90":[0.03225806451612903,0.2903225806451613]}}}},{"depth":"2","steps":21,"episodes":21,"metrics":{"type_match":{"raw":{"mean":0.42857142857142855,"ci90":[0.23809523809523808,0.6190476190476191]},"fog":{"mean":0.8095238095238095,"ci90":[0.6666666666666666,0.9523809523809523]},"fog_minus_raw":{"mean":0.38095238095238093,"ci90":[0.14285714285714285,0.6190476190476191]}},"exact_match":{"raw":{"mean":0.09523809523809523,"ci90":[0.0,0.19047619047619047]},"fog":{"mean":0.2857142857142857,"ci90":[0.14285714285714285,0.47619047619047616]},"fog_minus_raw":{"mean":0.19047619047619047,"ci90":[0.0,0.38095238095238093]}}}},{"depth":"3+","steps":26,"episodes":15,"metrics":{"type_match":{"raw":{"mean":0.34615384615384615,"ci90":[0.17391304347826086,0.4838709677419355]},"fog":{"mean":0.7692307692307693,"ci90":[0.6206896551724138,0.9411764705882353]},"fog_minus_raw":{"mean":0.4230769230769231,"ci90":[0.1723645320197045,0.7142857142857143]}},"exact_match":{"raw":{"mean":0.11538461538461539,"ci90":[0.037037037037037035,0.20689655172413793]},"fog":{"mean":0.5769230769230769,"ci90":[0.42424242424242425,0.7894736842105263]},"fog_minus_raw":{"mean":0.46153846153846156,"ci90":[0.25806451612903225,0.7142857142857143]}}}}],"temporal_reasoning":[{"depth":"0","steps":32,"episodes":32,"metrics":{"type_match":{"raw":{"mean":0.6875,"ci90":[0.5625,0.8125]},"fog":{"mean":0.9375,"ci90":[0.875,1.0]},"fog_minus_raw":{"mean":0.25,"ci90":[0.125,0.375]}},"exact_match":{"raw":{"mean":0.1875,"ci90":[0.09375,0.3125]},"fog":{"mean":0.46875,"ci90":[0.3125,0.625]},"fog_minus_raw":{"mean":0.28125,"ci90":[0.15625,0.40625]}}}},{"depth":"1","steps":32,"episodes":32,"metrics":{"type_match":{"raw":{"mean":0.5625,"ci90":[0.40625,0.71875]},"fog":{"mean":0.65625,"ci90":[0.5,0.78125]},"fog_minus_raw":{"mean":0.09375,"ci90":[-0.0625,0.25]}},"exact_match":{"raw":{"mean":0.15625,"ci90":[0.0625,0.28125]},"fog":{"mean":0.40625,"ci90":[0.25,0.5625]},"fog_minus_raw":{"mean":0.25,"ci90":[0.125,0.375]}}}},{"depth":"2","steps":27,"episodes":27,"metrics":{"type_match":{"raw":{"mean":0.3333333333333333,"ci90":[0.18518518518518517,0.48148148148148145]},"fog":{"mean":0.7407407407407407,"ci90":[0.5925925925925926,0.8888888888888888]},"fog_minus_raw":{"mean":0.4074074074074074,"ci90":[0.2222222222222222,0.5925925925925926]}},"exact_match":{"raw":{"mean":0.14814814814814814,"ci90":[0.037037037037037035,0.25925925925925924]},"fog":{"mean":0.6296296296296297,"ci90":[0.48148148148148145,0.7777777777777778]},"fog_minus_raw":{"mean":0.48148148148148145,"ci90":[0.3333333333333333,0.6296296296296297]}}}},{"depth":"3+","steps":33,"episodes":15,"metrics":{"type_match":{"raw":{"mean":0.24242424242424243,"ci90":[0.0967741935483871,0.3783783783783784]},"fog":{"mean":0.7272727272727273,"ci90":[0.6,0.8333333333333334]},"fog_minus_raw":{"mean":0.48484848484848486,"ci90":[0.375,0.6071428571428571]}},"exact_match":{"raw":{"mean":0.0,"ci90":[0.0,0.0]},"fog":{"mean":0.3939393939393939,"ci90":[0.2826086956521739,0.5185185185185185]},"fog_minus_raw":{"mean":0.3939393939393939,"ci90":[0.2857142857142857,0.5185185185185185]}}}}],"instant_response":[{"depth":"0","steps":23,"episodes":23,"metrics":{"type_match":{"raw":{"mean":0.7391304347826086,"ci90":[0.5652173913043478,0.8695652173913043]},"fog":{"mean":0.6521739130434783,"ci90":[0.4782608695652174,0.8260869565217391]},"fog_minus_raw":{"mean":-0.08695652173913043,"ci90":[-0.2608695652173913,0.08695652173913043]}},"exact_match":{"raw":{"mean":0.17391304347826086,"ci90":[0.043478260869565216,0.30434782608695654]},"fog":{"mean":0.2608695652173913,"ci90":[0.13043478260869565,0.43478260869565216]},"fog_minus_raw":{"mean":0.08695652173913043,"ci90":[0.0,0.17391304347826086]}}}},{"depth":"1","steps":23,"episodes":23,"metrics":{"type_match":{"raw":{"mean":0.782608695652174,"ci90":[0.6521739130434783,0.9130434782608695]},"fog":{"mean":0.6086956521739131,"ci90":[0.43478260869565216,0.782608695652174]},"fog_minus_raw":{"mean":-0.17391304347826086,"ci90":[-0.34782608695652173,0.0]}},"exact_match":{"raw":{"mean":0.13043478260869565,"ci90":[0.043478260869565216,0.2608695652173913]},"fog":{"mean":0.2608695652173913,"ci90":[0.13043478260869565,0.43478260869565216]},"fog_minus_raw":{"mean":0.13043478260869565,"ci90":[0.0,0.30434782608695654]}}}},{"depth":"2","steps":21,"episodes":21,"metrics":{"type_match":{"raw":{"mean":0.42857142857142855,"ci90":[0.23809523809523808,0.6190476190476191]},"fog":{"mean":0.7142857142857143,"ci90":[0.5714285714285714,0.8571428571428571]},"fog_minus_raw":{"mean":0.2857142857142857,"ci90":[0.047619047619047616,0.5238095238095238]}},"exact_match":{"raw":{"mean":0.047619047619047616,"ci90":[0.0,0.14285714285714285]},"fog":{"mean":0.38095238095238093,"ci90":[0.19047619047619047,0.5714285714285714]},"fog_minus_raw":{"mean":0.3333333333333333,"ci90":[0.19047619047619047,0.5238095238095238]}}}},{"depth":"3+","steps":15,"episodes":11,"metrics":{"type_match":{"raw":{"mean":0.26666666666666666,"ci90":[0.09090909090909091,0.46153846153846156]},"fog":{"mean":0.7333333333333333,"ci90":[0.5625,0.9230769230769231]},"fog_minus_raw":{"mean":0.4666666666666667,"ci90":[0.2857142857142857,0.6875]}},"exact_match":{"raw":{"mean":0.13333333333333333,"ci90":[0.0,0.3333333333333333]},"fog":{"mean":0.6,"ci90":[0.3888888888888889,0.8461538461538461]},"fog_minus_raw":{"mean":0.4666666666666667,"ci90":[0.26666666666666666,0.7142857142857143]}}}}]}''')

assert tuple(DATA) == tuple(category for category, _ in CATEGORY_ORDER)
assert all(
    tuple(row["depth"] for row in DATA[category]) == DEPTH_ORDER
    for category, _ in CATEGORY_ORDER
)
print(f"Loaded {len(DATA)} category panels × {len(DEPTH_ORDER)} depth bins with {CONFIDENCE_LEVEL:.0%} CIs.")


In [ ]:
# Convert the embedded nested data into a complete tidy table.
table_rows = []
for category, display_category in CATEGORY_ORDER:
    for depth_row in DATA[category]:
        for metric in ("type_match", "exact_match"):
            for method in ("raw", "fog"):
                values = depth_row["metrics"][metric][method]
                table_rows.append({
                    "category": category,
                    "display_category": display_category,
                    "depth": depth_row["depth"],
                    "steps": depth_row["steps"],
                    "episodes": depth_row["episodes"],
                    "metric": metric,
                    "method": method,
                    "mean": values["mean"],
                    "ci_low": values["ci90"][0],
                    "ci_high": values["ci90"][1],
                    "fog_minus_raw": depth_row["metrics"][metric]["fog_minus_raw"]["mean"],
                    "gain_ci_low": depth_row["metrics"][metric]["fog_minus_raw"]["ci90"][0],
                    "gain_ci_high": depth_row["metrics"][metric]["fog_minus_raw"]["ci90"][1],
                })

plot_df = pd.DataFrame(table_rows)
assert len(plot_df) == 6 * 4 * 2 * 2
assert not plot_df.isna().any().any()

# Show all 96 plotting rows rather than a truncated preview.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 180,
):
    print(plot_df.to_string(index=False))


In [ ]:
def plot_category_depth(metric, output_dir=OUTPUT_DIR):
    """Plot one 2×3 task-category group figure with 90% CIs and export it."""
    if metric not in METRIC_NAMES:
        raise ValueError(f"Unknown metric: {metric}")

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(13.2, 8.0),
        sharex=True,
        sharey=True,
    )

    for axis, (category, display_category) in zip(axes.flat, CATEGORY_ORDER):
        panel = plot_df[
            (plot_df["category"] == category)
            & (plot_df["metric"] == metric)
        ]
        x = np.arange(len(DEPTH_ORDER))

        for method in ("raw", "fog"):
            rows = (
                panel[panel["method"] == method]
                .set_index("depth")
                .loc[list(DEPTH_ORDER)]
            )
            means = rows["mean"].to_numpy(dtype=float)
            ci_low = rows["ci_low"].to_numpy(dtype=float)
            ci_high = rows["ci_high"].to_numpy(dtype=float)
            errors = np.vstack((means - ci_low, ci_high - means))

            axis.errorbar(
                x,
                means * 100.0,
                yerr=errors * 100.0,
                color=COLORS[method],
                marker="o",
                markersize=5.5,
                linewidth=2.0,
                capsize=2.5,
                label=METHOD_NAMES[method],
            )

        count_rows = (
            panel[panel["method"] == "raw"]
            .set_index("depth")
            .loc[list(DEPTH_ORDER)]
        )
        for position, steps in enumerate(count_rows["steps"].astype(int)):
            axis.text(
                position,
                2.0,
                f"n={steps}",
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="#4B5563",
            )

        axis.set_title(display_category, fontweight="bold")
        axis.set_xticks(x, DEPTH_ORDER)
        axis.set_ylim(0, 100)
        axis.grid(axis="y", color="#D1D5DB", linewidth=0.7, alpha=0.7)
        axis.spines[["top", "right"]].set_visible(False)

    axes[0, 0].set_ylabel("Per-step accuracy (%)")
    axes[1, 0].set_ylabel("Per-step accuracy (%)")
    for axis in axes[1, :]:
        axis.set_xlabel("Interaction step index")
    axes[0, 2].legend(loc="upper right", frameon=False)

    fig.suptitle(
        f"{METRIC_NAMES[metric]} by Task Category and Interaction Depth (90% CI)",
        fontsize=14,
        fontweight="bold",
    )
    fig.tight_layout()

    stem = (
        "fog_category_depth_type"
        if metric == "type_match"
        else "fog_category_depth_exact"
    )
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf", "svg"):
        fig.savefig(
            output_dir / f"{stem}.{suffix}",
            dpi=300,
            bbox_inches="tight",
        )

    return fig


In [ ]:
# Figure 1: Exact Action Match (recommended main-paper figure)
exact_figure = plot_category_depth("exact_match")
plt.show()
print(f"Exported Exact Match figure to: {OUTPUT_DIR.resolve()}")


In [ ]:
# Figure 2: Action Type Match (recommended supplementary figure)
type_figure = plot_category_depth("type_match")
plt.show()
print(f"Exported Type Match figure to: {OUTPUT_DIR.resolve()}")


In [ ]:
# Compact numerical table corresponding to the two figures.
gain_table = (
    plot_df[plot_df["method"] == "raw"]
    .pivot_table(
        index=["display_category", "metric"],
        columns="depth",
        values="fog_minus_raw",
        aggfunc="first",
    )
    .loc[
        [
            (display_category, metric)
            for _, display_category in CATEGORY_ORDER
            for metric in ("type_match", "exact_match")
        ],
        list(DEPTH_ORDER),
    ]
    * 100.0
)
gain_table.columns = [f"Step {column} gain (pp)" for column in gain_table.columns]
print(gain_table.round(2).to_string())
